Povoamento de Dados- Dimensão Cliente

Importação de Pacotes

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

Leitura de Dados da fonte

In [5]:
clientes = pd.read_csv('../Fontes/clientes_fonte.csv', encoding='latin1')
print(f"Registos lidos: {clientes.shape[0]}")
display(clientes.head())

Registos lidos: 100


,id_cliente,nome,email,password_sha256,altura_cm,peso_kg,profissao,data_nascimento
0,1,Rita Lima,tcardoso@example.net,9cdd87d170fb9227db6c6c4c8840a6a13e502fe1cbb202...,188,58,Estudante,9/18/1980
1,2,Salvador Vieira,erica10@example.org,a5523bffc5331c394dca52c608ecff4ce2dd3ae4af2473...,178,73,Cozinheiro,8/16/1987
2,3,Melanie Figueiredo,festeves@example.com,2fe1e7708b141e46e583c3b3092a92775655bb83d87908...,164,50,Designer,4/19/1967
3,4,Afonso Figueiredo,valentinacunha@example.org,4502c30db76b9285b4bae55acfef3eea13415de870020c...,192,93,Engenheiro,5/23/1998
4,5,Fernando Carvalho-Carneiro,paivasalome@example.com,4490df250e496158c6be42bc77eaedce4ecfa5226244bf...,157,57,Vendedor,10/9/1995


Limpeza e Validação: verificar valores duplos e nulos

In [6]:
print("Duplicados:", clientes.duplicated(subset=['id_cliente']).sum())
print("Nulos por coluna:\n", clientes.isnull().sum())

Duplicados: 0
Nulos por coluna:
 id_cliente         0
nome               0
email              0
password_sha256    0
altura_cm          0
peso_kg            0
profissao          0
data_nascimento    0
dtype: int64


Criação do Atributo Faixa Etária

In [7]:
def calcular_faixa_etaria(data_nascimento, ref_date=datetime(2025,9,1)):
    try:
        birth = datetime.strptime(data_nascimento, '%m/%d/%Y')
        idade = (ref_date - birth).days // 365
        if idade < 18:
            return 'Menor de 18'
        elif idade < 30:
            return '18-29'
        elif idade < 45:
            return '30-44'
        elif idade < 60:
            return '45-59'
        else:
            return '60+'
    except:
        return 'Desconhecido'

clientes['faixa_etaria'] = clientes['data_nascimento'].apply(calcular_faixa_etaria)

Verificação da Integridade 

In [8]:
print(clientes['faixa_etaria'].value_counts())
assert clientes['id_cliente'].is_unique, "IDs de cliente não são únicos!"
assert clientes['faixa_etaria'].isnull().sum() == 0, "Existem clientes sem faixa etária!"

faixa_etaria
30-44    34
45-59    30
18-29    18
60+      18
Name: count, dtype: int64


Seleção de Colunas para o DW

In [9]:
dim_cliente = clientes[['id_cliente', 'nome', 'altura_cm', 'peso_kg', 'profissao', 'data_nascimento', 'faixa_etaria']].copy()
dim_cliente = dim_cliente.rename(columns={
    'id_cliente': 'cliente_id',
    'nome': 'cliente_nome',
    'altura_cm': 'cliente_altura_cm',
    'peso_kg': 'cliente_peso_kg',
    'profissao': 'cliente_profissao',
    'data_nascimento': 'cliente_data_nascimento',
    'faixa_etaria': 'cliente_faixa_etaria'
})
dim_cliente.insert(1, 'cliente_sk', range(1, 1 + len(dim_cliente)))

Exportação dos Dados

In [10]:
dim_cliente.to_csv("../Dados Finais/dim_cliente.csv", index=False, encoding="utf-8-sig")
print("Dimensão cliente pronta para carga!")
display(dim_cliente.head())

Dimensão cliente pronta para carga!


,cliente_id,cliente_sk,cliente_nome,cliente_altura_cm,cliente_peso_kg,cliente_profissao,cliente_data_nascimento,cliente_faixa_etaria
0,1,1,Rita Lima,188,58,Estudante,9/18/1980,30-44
1,2,2,Salvador Vieira,178,73,Cozinheiro,8/16/1987,30-44
2,3,3,Melanie Figueiredo,164,50,Designer,4/19/1967,45-59
3,4,4,Afonso Figueiredo,192,93,Engenheiro,5/23/1998,18-29
4,5,5,Fernando Carvalho-Carneiro,157,57,Vendedor,10/9/1995,18-29
